# 02 — Player seasons

Build the clean, consolidated `player_season` table (2011-12 → 2025-26) from the cached raw responses in `data/raw/`.

## Step 2A — Raw data inventory

`python -m unicorn.download` populates `data/raw/` (703 requests, resumable). Re-running it here costs nothing because every request is already cached. Before any cleaning, check the raw layer is complete and plausible.

In [1]:
import pandas as pd
import numpy as np
from unicorn.download import download_all

log = download_all()   # instant if already downloaded
print(log["status"].str.split(":").str[0].value_counts().to_dict())
log[log["status"].str.startswith("FAILED")]

{'cached': 733}


,endpoint,season,season_type_all_star,measure_type_detailed_defense,per_mode_detailed,rows,status,starter_bench_nullable,distance_range,per_mode_simple,team_id,season_all_time


In [2]:
# Rows per season and endpoint: player counts should be stable (~450-600 regular season)
lds = log[log["endpoint"] == "LeagueDashPlayerStats"]
lds.pivot_table(index="season", columns=["season_type_all_star", "measure_type_detailed_defense"],
                values="rows", aggfunc="first")

season_type_all_star          Playoffs                                  \
measure_type_detailed_defense Advanced Base Defense Misc Scoring Usage   
season                                                                   
2011-12                            204  204     204  204     204   204   
2012-13                            209  209     209  209     209   209   
2013-14                            204  204     204  204     204   204   
2014-15                            208  208     208  208     208   208   
2015-16                            215  215     215  215     215   215   
2016-17                            215  215     215  215     215   215   
2017-18                            210  210     210  210     210   210   
2018-19                            212  212     212  212     212   212   
2019-20                            217  217     217  217     217   217   
2020-21                            239  239     239  239     239   239   
2021-22                            217  217     217  217     217   217   
2022-23                            217  217     217  217     217   217   
2023-24                            214  214     214  214     214   214   
2024-25                            219  219     219  219     219   219   
2025-26                            230  230     230  230     230   230   

season_type_all_star          Regular Season                                  
measure_type_detailed_defense       Advanced Base Defense Misc Scoring Usage  
season                                                                        
2011-12                                  478  478     478  478     478   478  
2012-13                                  469  469     469  469     469   469  
2013-14                                  482  482     482  482     482   482  
2014-15                                  492  492     492  492     492   492  
2015-16                                  476  476     476  476     476   476  
2016-17                                  486  486     486  486     486   486  
2017-18                                  540  540     540  540     540   540  
2018-19                                  530  530     530  530     530   530  
2019-20                                  529  529     529  529     529   529  
2020-21                                  540  540     540  540     540   540  
2021-22                                  605  605     605  605     605   605  
2022-23                                  539  539     539  539     539   539  
2023-24                                  572  572     572  572     572   572  
2024-25                                  569  569     569  569     569   569  
2025-26                                  582  582     582  582     582   582

In [3]:
other = (log[log["endpoint"].isin(["LeagueDashPlayerShotLocations", "LeagueDashPlayerBioStats", "CommonTeamRoster"])]
         .assign(season_type=lambda d: d["season_type_all_star"].fillna("—"))
         .pivot_table(index="season", columns=["endpoint", "season_type"], values="rows", aggfunc="sum"))
other

endpoint    CommonTeamRoster LeagueDashPlayerBioStats  \
season_type                —                        —   
season                                                  
2011-12                  647                      478   
2012-13                  635                      469   
2013-14                  651                      482   
2014-15                  661                      492   
2015-16                  621                      476   
2016-17                  637                      486   
2017-18                  676                      540   
2018-19                  678                      530   
2019-20                  737                      529   
2020-21                  738                      540   
2021-22                  766                      605   
2022-23                  765                      539   
2023-24                  770                      572   
2024-25                  797                      569   
2025-26                  813                      582   

endpoint    LeagueDashPlayerShotLocations                 
season_type                      Playoffs Regular Season  
season                                                    
2011-12                               203            478  
2012-13                               209            469  
2013-14                               204            482  
2014-15                               208            492  
2015-16                               215            476  
2016-17                               215            486  
2017-18                               210            540  
2018-19                               212            530  
2019-20                               215            529  
2020-21                               239            540  
2021-22                               217            605  
2022-23                               217            539  
2023-24                               214            572  
2024-25                               219            569  
2025-26                               230            582

In [4]:
combine = log[log["endpoint"] == "DraftCombineStats"].set_index("season_all_time")["rows"]
print("combine players per draft year:\n", combine.to_string())
print("\nDraftHistory rows:", log.loc[log["endpoint"] == "DraftHistory", "rows"].item())

combine players per draft year:
 season_all_time
2000-01    65
2001-02    78
2002-03    82
2003-04    78
2004-05    81
2005-06    81
2006-07    76
2007-08    77
2008-09    76
2009-10    46
2010-11    52
2011-12    54
2012-13    61
2013-14    63
2014-15    56
2015-16    58
2016-17    56
2017-18    64
2018-19    69
2019-20    77
2020-21    45
2021-22    74
2022-23    83
2023-24    81
2024-25    83
2025-26    79
2026-27    78

DraftHistory rows: 8434


### Roster coverage (listed position, decision D4)

The roster row counts above include the coaches table. What matters is: for each season, what share of player-seasons (and of minutes) get a listed position from that season's rosters?

In [5]:
from nba_api.stats.endpoints import commonteamroster, leaguedashplayerstats
from nba_api.stats.static import teams
from unicorn.download import SEASONS
from unicorn.raw import fetch

def season_rosters(season):
    return pd.concat([fetch(commonteamroster.CommonTeamRoster, team_id=t["id"], season=season)["CommonTeamRoster"]
                      for t in teams.get_teams()], ignore_index=True)

def season_base(season):
    return fetch(leaguedashplayerstats.LeagueDashPlayerStats, season=season, season_type_all_star="Regular Season",
                 measure_type_detailed_defense="Base", per_mode_detailed="Totals")["LeagueDashPlayerStats"]

rows = []
for s in SEASONS:
    ros, base = season_rosters(s), season_base(s)
    has_pos = base["PLAYER_ID"].isin(ros.loc[ros["POSITION"].fillna("") != "", "PLAYER_ID"])
    rows.append({"season": s, "players": len(base), "roster_players": ros["PLAYER_ID"].nunique(),
                 "dup_roster_ids": ros["PLAYER_ID"].duplicated().sum(),
                 "pct_players_with_pos": has_pos.mean(),
                 "pct_minutes_with_pos": base.loc[has_pos, "MIN"].sum() / base["MIN"].sum()})
coverage = pd.DataFrame(rows).set_index("season")
coverage.style.format({"pct_players_with_pos": "{:.1%}", "pct_minutes_with_pos": "{:.1%}"})

,players,roster_players,dup_roster_ids,pct_players_with_pos,pct_minutes_with_pos
season,,,,,
2011-12,478,442,0,92.1%,98.7%
2012-13,469,446,0,93.0%,99.0%
2013-14,482,448,0,90.9%,98.8%
2014-15,492,450,0,90.9%,98.8%
2015-16,476,450,1,93.3%,98.7%
2016-17,486,451,0,91.8%,98.5%
2017-18,540,502,0,91.5%,98.8%
2018-19,530,497,0,91.1%,98.6%
2019-20,529,510,0,94.3%,99.2%


### Step 2A findings

- **All 703 requests are cached with no failures.** The six measure types agree exactly on player counts in every season and season type, so they can be joined one-to-one on `PLAYER_ID`.
- **Regular-season player counts** rise from about 470 (2011-14) to about 570 (2023-26). This reflects two-way contracts (from 2017-18) and larger rosters. **2021-22 has 605** because of COVID health-and-safety replacement signings.
- **Shot locations** occasionally have 1–2 fewer players than Base (2011-12 and 2019-20 playoffs): almost certainly players with zero FGA. We'll left-join and treat missing as 0 attempts.
- **Roster position coverage:** about 91% of player-seasons and about 99% of minutes. The low point is 2021-22 (81.5% of players, 97.5% of minutes), when the COVID call-ups had left rosters by the snapshot date. 2015-16 has one player on two rosters, so deduplicate when joining.
- **Combine:** 45–83 players per draft year. Dips in 2009-10 and 2020-21 (the 2020 combine was virtual).
- **Shortened seasons** to remember for counting stats: 2011-12 (66 games, lockout), 2019-20 (suspension + bubble), 2020-21 (72 games). Rate stats are unaffected; totals and games-played thresholds must be relative to team games.

## Step 2B — Build the clean `player_season` tables

Built by `unicorn.player_season.build_player_seasons()` from the raw cache. Decisions applied:

- **D1** One consolidated row per player-season. Traded players keep `team_count`; `team_abbreviation` = last team.
- **D3** Height and weight as listed that season.
- **D4** Position as listed on that season's roster (`position_listed`). Gaps are filled **only from the player's most recent earlier season** (`position`, with `position_source` showing which). `position_num` maps G=1 … C=5 for the derived-position work (D5).
- **D6** `age` = exact age on **Feb 1** of the season, from roster birth date. Birth date is a fixed fact, so it can come from any season's roster. Fallback: the NBA's integer age (`age_source`).
- **D7** **No rows dropped.** Reliability columns (`min`, `poss`, `fga`, `fta`, `fg3a`) are kept so minutes thresholds can be toggled later.
- **D8** **Playoffs in a separate table** with the same key (stats only, no bio).
- **Draft info from `DraftHistory`** (keyed on player ID), not from bio. The bio endpoint carries wrong records for some players, e.g. Walker Russell Jr. gets his father's 1982 pick. `undrafted` = absent from DraftHistory.
- All percentages are recomputed from makes and attempts, so 0 attempts → NaN. `gs` comes from the "Starters" filter (GP when starting, verified against `PlayerCareerStats`).

In [6]:
from unicorn.player_season import build_player_seasons

ps = build_player_seasons("Regular Season")
po = build_player_seasons("Playoffs")
print("regular season:", ps.shape, "| playoffs:", po.shape)

regular season: (7889, 113) | playoffs: (3230, 98)


### Integrity checks

In [7]:
raw_counts = log[(log["endpoint"] == "LeagueDashPlayerStats") & (log["measure_type_detailed_defense"] == "Base")
                 & log["starter_bench_nullable"].isna()].pivot_table(index="season", columns="season_type_all_star", values="rows")

checks = {
    "unique (player_id, season) — RS": not ps.duplicated(["player_id", "season"]).any(),
    "unique (player_id, season) — PO": not po.duplicated(["player_id", "season"]).any(),
    "RS rows match raw per season": ps.groupby("season").size().equals(raw_counts["Regular Season"].astype(int)),
    "PO rows match raw per season": po.groupby("season").size().equals(raw_counts["Playoffs"].astype(int)),
    "gs <= gp": (ps["gs"] <= ps["gp"]).all(),
    "fgm <= fga, fg3a <= fga": ((ps["fgm"] <= ps["fga"]) & (ps["fg3a"] <= ps["fga"])).all(),
    "drafted no later than the season played": (ps["draft_year"].isna() | (ps["draft_year"] <= ps["season_start"])).all(),
}
pd.Series(checks)

unique (player_id, season) — RS            True
unique (player_id, season) — PO            True
RS rows match raw per season               True
PO rows match raw per season               True
gs <= gp                                   True
fgm <= fga, fg3a <= fga                    True
drafted no later than the season played    True
dtype: bool

In [8]:
# Playoff-only seasons: players signed after the regular season ended
# (e.g. Tracy McGrady → Spurs 2013, Dorell Wright → Heat 2016 after playing in China). Real, not a join error.
po_only = po[~po.set_index(["player_id", "season"]).index.isin(ps.set_index(["player_id", "season"]).index)]
print(len(po_only), "playoff-only player-seasons")
po_only[["player_name", "season", "gp", "min"]]

9 playoff-only player-seasons


,player_name,season,gp,min
214,Tracy McGrady,2012-13,6,31.263333
297,Dwayne Jones,2012-13,2,1.895000
866,Dorell Wright,2015-16,5,18.910000
1017,John Holland,2015-16,1,1.316667
1315,Ty Lawson,2017-18,5,96.300000
1862,Jaylen Adams,2019-20,3,20.733333
2334,Luca Vildoza,2021-22,7,16.936667
2377,Tristan Thompson,2022-23,6,32.341667
2502,DaQuan Jeffries,2022-23,2,4.588333


In [9]:
# Shot zones should account for (nearly) every field-goal attempt. 'corner3' is the sum of
# left + right corner, so zones are: RA + paint + mid + corner3 + atb3 + backcourt.
zone_fga = ps[["ra_fga", "paint_non_ra_fga", "mid_fga", "corner3_fga", "atb3_fga", "backcourt_fga"]].sum(axis=1)
gap = ps["fga"] - zone_fga
print("player-seasons where zones == FGA:", f"{(gap == 0).mean():.1%}")
print("total FGA unaccounted for:", f"{gap.sum() / ps['fga'].sum():.3%}")
gap.describe().round(2)

player-seasons where zones == FGA: 96.0%
total FGA unaccounted for: 0.015%


count    7889.00
mean        0.06
std         0.37
min         0.00
25%         0.00
50%         0.00
75%         0.00
max        10.00
dtype: float64

In [10]:
# Missingness: which columns have gaps, and is it explicable?
na = ps.isna().mean()
na[na > 0].sort_values(ascending=False).to_frame("share_missing").style.format("{:.1%}")

,share_missing
draft_round,22.8%
draft_year,22.8%
draft_number,22.8%
position_listed,8.9%
exp,8.9%
fg3_pct,8.1%
ft_pct,4.6%
position,3.0%
position_num,3.0%
birth_date,2.3%


In [11]:
print(ps["position_source"].value_counts().to_dict())
print(ps["age_source"].value_counts().to_dict())
# The share of *minutes* without any position is what matters for modelling
print("minutes with no position:", f"{ps.loc[ps['position_source'] == 'missing', 'min'].sum() / ps['min'].sum():.2%}")

{'roster': 7189, 'prior_season': 465, 'missing': 235}
{'birth_date': 7708, 'nba_integer': 181}
minutes with no position: 0.24%


### Sanity check: Shai's development

In [12]:
cols = ["season", "team_abbreviation", "age", "height_in", "weight_lb", "position", "gp", "gs", "min",
        "pts_per36", "ast_pct", "usg_pct", "ts_pct", "rim_rate", "fg3a_rate", "pct_uast_fgm"]
ps.loc[ps["player_name"] == "Shai Gilgeous-Alexander", cols].round(3)

,season,team_abbreviation,age,height_in,weight_lb,position,gp,gs,min,pts_per36,ast_pct,usg_pct,ts_pct,rim_rate,fg3a_rate,pct_uast_fgm
3882,2018-19,LAC,20.559,78.0,181.0,G,82,73,2173.907,14.722,0.179,0.182,0.554,0.353,0.194,0.601
4325,2019-20,OKC,21.558,77.0,180.0,G-F,70,70,2428.335,19.732,0.148,0.235,0.568,0.360,0.247,0.709
4804,2020-21,OKC,22.560,78.0,180.0,G-F,35,35,1179.608,25.330,0.311,0.283,0.623,0.395,0.301,0.871
5316,2021-22,OKC,23.559,78.0,180.0,G,56,56,1941.855,25.417,0.293,0.298,0.557,0.299,0.282,0.832
5857,2022-23,OKC,24.559,78.0,195.0,G,68,68,2415.985,31.813,0.249,0.318,0.626,0.361,0.122,0.790
6361,2023-24,OKC,25.558,78.0,195.0,G,75,75,2552.687,31.788,0.287,0.317,0.636,0.303,0.181,0.760
6900,2024-25,OKC,26.560,78.0,195.0,G,76,76,2597.630,34.425,0.299,0.336,0.637,0.255,0.263,0.762
7438,2025-26,OKC,27.559,78.0,195.0,G,68,68,2258.947,33.738,0.332,0.323,0.665,0.276,0.226,0.803


### Save

In [13]:
from unicorn.raw import PROJECT_ROOT

out = PROJECT_ROOT / "data" / "processed"
ps.to_parquet(out / "player_season.parquet", index=False)
po.to_parquet(out / "player_season_playoffs.parquet", index=False)
for f in sorted(out.glob("*.parquet")):
    print(f.name, f"{f.stat().st_size / 1e6:.1f} MB")

player_season.parquet 2.4 MB
player_season_playoffs.parquet 0.8 MB


### Step 2B findings

- **`player_season`: 7,889 rows × 113 columns; `player_season_playoffs`: 3,230 × 98.** Unique keys, row counts identical to raw for every season.
- **Shot zones reconcile with FGA**: exact for 96% of player-seasons; 0.015% of all attempts unaccounted for (a few unclassified shots).
- **Position**: 7,189 rows from that season's roster, 465 filled from an earlier season, 235 missing (0.24% of minutes).
- **Age**: exact from birth date for 7,708 rows; NBA integer fallback for 181.
- **Data errors found and handled**: bio draft records wrong for some players (fixed by using `DraftHistory`); undrafted players are sometimes coded with blank or 0 picks rather than "Undrafted".
- **9 playoff-only player-seasons** (signed after the regular season). Real; they matter for the later "early playoff experience" analysis.
- **Early signal worth remembering**: Shai's unassisted share of made FGs went 0.60 → 0.71 → 0.87 (2018-19 → 2020-21) *before* his scoring breakout. Self-creation is a strong candidate breakout feature.